In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt

DATA_FOLDER = "Clean_Dataset"
HOUSE_PREFIX = "House_"
TOTAL_HOUSES = 13
OUTPUT_CSV_FOLDER = "nan_reports"
OUTPUT_PLOT_FOLDER = "nan_plots"
os.makedirs(OUTPUT_CSV_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_PLOT_FOLDER, exist_ok=True)

def find_nan_ranges(series):
    is_nan = series.isna()
    nan_ranges = []
    current_start = None
    for timestamp, val in is_nan.items():
        if val and current_start is None:
            current_start = timestamp
        elif not val and current_start is not None:
            nan_ranges.append((current_start, timestamp))
            current_start = None
    if current_start is not None:
        nan_ranges.append((current_start, series.index[-1]))
    return nan_ranges

for i in range(1, TOTAL_HOUSES + 1):
    house_id = f"{HOUSE_PREFIX}{i:02d}"
    house_path = os.path.join(DATA_FOLDER, house_id)
    report_rows = []

    print(f"\nProcessing {house_id}...")

    # === Electric Data ===
    electric_path = os.path.join(house_path, "Electric_data")
    p_agg_nans = pd.Series(dtype="bool")

    for fname in sorted(os.listdir(electric_path)):
        if fname.endswith(".csv") and "appliances_metadata" not in fname:
            fpath = os.path.join(electric_path, fname)
            df = pd.read_csv(fpath, parse_dates=["timestamp"])
            df.set_index("timestamp", inplace=True)
            if "P_agg" in df.columns:
                nan_ranges = find_nan_ranges(df["P_agg"])
                for start, end in nan_ranges:
                    report_rows.append(["Electric_data", fname, "P_agg", start, end])
                p_agg_part = df["P_agg"].isna()
                if not p_agg_part.empty:
                    p_agg_nans = pd.concat([p_agg_nans, p_agg_part])

    # === Environmental Data ===
    env_path = os.path.join(house_path, "Environmental_data")
    env_nans = {}

    for fname in sorted(os.listdir(env_path)):
        if fname.endswith(".csv"):
            fpath = os.path.join(env_path, fname)
            df = pd.read_csv(fpath, parse_dates=["timestamp"])
            df.set_index("timestamp", inplace=True)
            for col in df.columns:
                nan_ranges = find_nan_ranges(df[col])
                for start, end in nan_ranges:
                    report_rows.append(["Environmental_data", fname, col, start, end])
                nan_series = df[col].isna()
                if col not in env_nans:
                    env_nans[col] = nan_series
                else:
                    env_nans[col] = pd.concat([env_nans[col], nan_series])

    # === Save Report CSV ===
    report_df = pd.DataFrame(report_rows, columns=["Folder", "File", "Column", "NaN_Start", "NaN_End"])
    report_df.to_csv(os.path.join(OUTPUT_CSV_FOLDER, f"NaN_Report_{house_id}.csv"), index=False)

    # === Plot P_agg NaN Scatter ===
    if not p_agg_nans.empty:
        nan_series = p_agg_nans[p_agg_nans].astype(int)
        nan_percent = 100 * nan_series.sum() / len(p_agg_nans)
        timestamps = nan_series.index[::10]  # υποδειγματοληψία ανά 10 βήματα
        values = nan_series.values[::10]

        plt.figure(figsize=(12, 3))
        plt.scatter(timestamps, values, s=1, color="red", label="P_agg NaN")
        plt.title(f"{house_id} - P_agg NaN Scatter Plot ({nan_percent:.2f}%)")
        plt.yticks([0, 1])
        plt.ylabel("NaN")
        plt.grid(axis='x', linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_PLOT_FOLDER, f"{house_id}_P_agg_nan.png"))
        plt.close()

    # === Plot Environmental Features NaN Scatter ===
    if env_nans:
        plt.figure(figsize=(12, 5))
        for col, series in env_nans.items():
            nan_series = series[series].astype(int)
            nan_percent = 100 * nan_series.sum() / len(series)
            timestamps = nan_series.index[::10]  # υποδειγματοληψία
            values = nan_series.values[::10]
            plt.scatter(timestamps, values, s=1, label=f"{col} ({nan_percent:.2f}%)")

        plt.title(f"{house_id} - Environmental NaN Scatter Plot")
        plt.yticks([0, 1])
        plt.ylabel("NaN")
        plt.legend(markerscale=6, fontsize="small")
        plt.grid(axis='x', linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_PLOT_FOLDER, f"{house_id}_env_nan.png"))
        plt.close()

    input(f"➤ Πάτα Enter για να συνεχίσουμε με το επόμενο σπίτι...\n")



Processing House_01...


C:\Users\G\AppData\Local\Temp\ipykernel_8752\3081578524.py:49: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  p_agg_nans = pd.concat([p_agg_nans, p_agg_part])


➤ Πάτα Enter για να συνεχίσουμε με το επόμενο σπίτι...
 



Processing House_02...


C:\Users\G\AppData\Local\Temp\ipykernel_8752\3081578524.py:49: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  p_agg_nans = pd.concat([p_agg_nans, p_agg_part])


...

C:\Users\G\AppData\Local\Temp\ipykernel_8752\3081578524.py:49: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  p_agg_nans = pd.concat([p_agg_nans, p_agg_part])


➤ Πάτα Enter για να συνεχίσουμε με το επόμενο σπίτι...
 



Processing House_13...


C:\Users\G\AppData\Local\Temp\ipykernel_8752\3081578524.py:49: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  p_agg_nans = pd.concat([p_agg_nans, p_agg_part])


➤ Πάτα Enter για να συνεχίσουμε με το επόμενο σπίτι...
 
